In [1]:
import pandas as pd
pd.set_option("display.max_columns", 50)

amr = pd.read_csv("../data/raw/amr_ecoli.tsv", sep="\t", dtype=str)
print(amr.shape)
amr.head()

(489375, 10)


,genome_id,genome_name,taxon_id,antibiotic,resistant_phenotype,measurement,measurement_value,measurement_unit,laboratory_typing_method,computational_method
0,562.193047,Escherichia coli ECBSAC210134367,562,ciprofloxacin,Resistant,NaN,NaN,NaN,NaN,SIR XGBoost Model
1,562.227303,Escherichia coli PNUSAE191092,562,ciprofloxacin,Susceptible,NaN,NaN,NaN,NaN,SIR XGBoost Model
2,562.66875,Escherichia coli strain JS316,562,ciprofloxacin,NaN,8.0,8.0,mg/L,NaN,MIC XGBoost Model
3,562.36249,Escherichia coli strain CP53,562,ciprofloxacin,Susceptible,NaN,NaN,NaN,NaN,SIR XGBoost Model
4,562.205012,Escherichia coli PNUSAE009306,562,ciprofloxacin,NaN,0.25,0.25,mg/L,NaN,MIC XGBoost Model


In [2]:
#antibiotice si genome
print(amr["genome_id"].nunique(), "genome distincte")
amr["antibiotic"].value_counts()

114985 genome distincte


antibiotic
ciprofloxacin    246737
ampicillin       242638
Name: count, dtype: int64

In [3]:
#lab sau predictie?
print(amr["laboratory_typing_method"].value_counts(dropna=False))
print(amr["computational_method"].value_counts(dropna=False))

laboratory_typing_method
NaN               460567
Disk diffusion     13349
Broth dilution      8130
MIC                 6235
Agar dilution       1094
Name: count, dtype: int64
computational_method
SIR XGBoost Model    229796
MIC XGBoost Model    229734
NaN                   29845
Name: count, dtype: int64


In [4]:
#doar rezultate de laborator
lab = amr[amr["computational_method"].isna()].copy()
print(len(amr), "->", len(lab), "randuri dupa excluderea predictiilor")
print(lab.groupby("antibiotic")["resistant_phenotype"].value_counts())

489375 -> 29845 randuri dupa excluderea predictiilor
antibiotic     resistant_phenotype
ampicillin     Resistant              4546
               Susceptible            3008
               Intermediate             14
ciprofloxacin  Susceptible            6734
               Resistant              2740
               Intermediate            156
Name: count, dtype: int64


In [5]:
#dplicate
print("duplicate genom x antibiotic:", lab.duplicated(["genome_id", "antibiotic"]).sum())

duplicate genom x antibiotic: 9565


In [6]:
#metadate si coloana de an
meta = pd.read_csv("../data/raw/genome_meta_ecoli.tsv", sep="\t", dtype=str)
print(meta.shape)
[c for c in meta.columns if "year" in c or "date" in c or "country" in c]

(114985, 121)


['completion_date',
 'date_inserted',
 'date_modified',
 'isolation_country',
 'collection_date',
 'collection_year']

In [7]:
#imbinare si tabelul an × clasa (inlocuieste collection_year cu numele real din celula 6)
an = "collection_year"
cip = lab[lab["antibiotic"] == "ciprofloxacin"].merge(
    meta[["genome_id", an]], on="genome_id", how="left")
cip[an] = pd.to_numeric(cip[an], errors="coerce")
print("fara an:", cip[an].isna().sum(), "din", len(cip))
pd.crosstab(cip[an], cip["resistant_phenotype"])

fara an: 4812 din 16972


resistant_phenotype,Intermediate,Resistant,Susceptible
collection_year,,,
1970.0,0,0,1
1977.0,0,0,1
1994.0,0,0,1
1997.0,0,0,1
1998.0,0,0,3
1999.0,0,1,6
2000.0,0,2,2
2001.0,0,17,136
2002.0,1,22,270
